# 03 — Comparaison des 3 dununs (V2)

Qu'est-ce qui distingue le dunun de **Manjanin**, **Maraka** et **Woloso** ? Le dunun n'ayant **qu'un seul type de frappe**, on compare *où* et *quand* il frappe (motif, densité, micro-timing, tempo), et le rôle des deux dununs (D1/D2).

> Rappel : le dunun (grave) est la voix qui **signe** le rythme — l'accompagnement jembé, lui, est partagé entre les pièces.

## 0. Imports & helpers

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score
import utils_V2 as u

df = pd.read_csv('output/dunun_frappes_V2.csv'); dd = u.list_dunun()
d1 = df[df['code']=='D1']
COL = dict(zip(u.PIECES, ['#4a8cff', '#e0952e', '#2fbf6e']))

def profs(p, code='D1'):
    out = [pr.values for t in dd[(dd['piece']==p)&(dd['code']==code)]['take'].unique()
           if (pr := u.subdivision_profile(p, int(t), code)) is not None]
    return np.array(out)
def motif(p, code='D1'): return profs(p, code).mean(0)

## 1. Le motif — où le dunun frappe dans le cycle
La signature rythmique. Manjanin est épuré, Maraka dense, Woloso balancé (probas ~0.5).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for p in u.PIECES:
    ax[0].step(range(1,13), motif(p), where='mid', lw=2.3, color=COL[p], label=p)
ax[0].set(title='Motif du dunun (proba de frappe)', xlabel='subdivision', ylim=(-.05,1.15)); ax[0].set_xticks(range(1,13)); ax[0].legend()
M = np.vstack([motif(p) for p in u.PIECES])
im = ax[1].imshow(M, aspect='auto', cmap='magma', vmin=0, vmax=1)
ax[1].set(title='Vue compacte', xticks=range(12), xticklabels=range(1,13), yticks=range(3), yticklabels=u.PIECES)
fig.colorbar(im, ax=ax[1], label='proba'); plt.tight_layout(); plt.show()

## 2. Densité — combien de frappes par cycle

In [ ]:
dens = {p: motif(p).sum() for p in u.PIECES}
plt.figure(figsize=(6,3.5))
plt.bar(dens.keys(), dens.values(), color=[COL[p] for p in u.PIECES])
for p,v in dens.items(): plt.text(p, v+0.05, f'{v:.1f}', ha='center')
plt.ylabel('frappes / cycle'); plt.title('Densité du dunun'); plt.show()

## 3. Micro-timing — le swing du dunun
Écart entre la position réelle des frappes et la grille parfaite (isochrone). Le dunun aussi est joué « swingué ».

In [ ]:
iso = np.arange(12)/12
plt.figure(figsize=(13,4))
for p in u.PIECES:
    m = d1[d1['piece']==p].groupby('SD')['rel'].mean().reindex(range(1,13))
    plt.plot(range(1,13), (m.values - iso)*100, marker='o', lw=2, color=COL[p], label=p)
plt.axhline(0, color='k', lw=1, alpha=.4)
plt.xticks(range(1,13)); plt.xlabel('subdivision'); plt.ylabel('écart à la grille (% du cycle)')
plt.title('Micro-timing du dunun'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 4. Tempo, D1 vs D2 sur prises appariées, et séparabilité

In [ ]:
print('ACCELERANDO (durée de cycle début→fin) :')
for p in u.PIECES:
    accs = []
    for t in dd[dd['piece']==p]['take'].unique():
        _, du = u.cycle_durations(p, int(t))
        if len(du) > 10: accs.append((du[:6].mean()/du[-6:].mean()-1)*100)
    print(f'  {p:9s}: +{np.mean(accs):.0f}%')

print('\nD1 vs D2 — seulement les prises appariées disponibles :')
for r in u.paired_dunun_takes().itertuples(index=False):
    p, t = r.piece, int(r.take)
    p1, p2 = u.subdivision_profile(p,int(t),'D1'), u.subdivision_profile(p,int(t),'D2')
    if p1 is not None and p2 is not None:
        print(f'  {p}_{t}: D1={p1.values.sum():.0f}/cyc, D2={p2.values.sum():.0f}/cyc')

rows = [{'piece':p, **{f's{i}':u.subdivision_profile(p,int(t),'D1').values[i] for i in range(12)}}
        for p in u.PIECES for t in dd[(dd['piece']==p)&(dd['code']=='D1')]['take'].unique()
        if u.subdivision_profile(p,int(t),'D1') is not None]
R = pd.DataFrame(rows); F = [c for c in R if c.startswith('s')]
pred = cross_val_predict(RandomForestClassifier(300, random_state=0), R[F].values, R['piece'].values, cv=LeaveOneOut())
print(f'\nSÉPARABILITÉ — rythme reconnu par le motif du dunun : {accuracy_score(R["piece"], pred):.0%} (leave-one-out, {len(R)} prises)')
display(pd.crosstab(R['piece'], pred))

## Synthèse — la carte d'identité des 3 dununs

| | Manjanin | Maraka | Woloso |
|---|---|---|---|
| **Motif** | épuré, sur les temps (1,3,6,9,11) | dense, étalé | balancé (probas ~0.5) |
| **Densité** | 4.9 /cyc | **5.6** /cyc | 4.6 /cyc |
| **Micro-timing** | swingué | swingué (autre profil) | swingué |
| **Accelerando** | +42 % | +49 % | **+54 %** |
| **2ᵉ dunun (D2)** | partiel : prises 1,4 | partiel : prise 6 | partiel : prise 2 |

**À retenir** : le dunun distingue les 3 rythmes (**86 %** au seul motif). C'est le *où* et le *quand* (motif + timing) qui font la différence — pas un type de frappe (le dunun n'en a qu'un). Suite : `04_dunun_visu_V2` (roues + timelines animées, colorées par D1/D2).